In [ ]:
import pandas as pd
df=pd.read_parquet(r'D:\AI\Real projects\Academic_Advisor\data\final\without_)outliers.parquet')

In [ ]:
pd.set_option('display.max_columns',60)
pd.set_option('display.max_rows',70)

In [ ]:
df.head()

In [ ]:
df[(df['prev_gpa_points_clean']==0)&(df['start_level_ord']==3)]

In [ ]:
df1=pd.read_parquet(r'D:\AI\Real projects\Academic_Advisor\data\raw\v_add_student_degree_status.parquet')

In [ ]:
df['student_id'].nunique()

In [ ]:
df.columns

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# 0. START FROM CLEAN DATAFRAME
# ============================================================

df_model = df.copy()

# Keep only normal rows for Model A
if "exclude_over_policy_semester" in df_model.columns:
    df_model = df_model[df_model["exclude_over_policy_semester"].eq(False)].copy()

print("Rows after removing over-policy semesters:", len(df_model))


# ============================================================
# 1. PREPARE part_id AND FILTER MODELING PERIOD
# ============================================================

df_model["_part_id_num"] = pd.to_numeric(df_model["part_id"], errors="coerce")

# Drop rows with invalid part_id
df_model = df_model[df_model["_part_id_num"].notna()].copy()
df_model["_part_id_num"] = df_model["_part_id_num"].astype(int)

# Modeling period decision
MODELING_START_PART_ID = 20051
MODELING_END_PART_ID = 20251

df_model = df_model[
    df_model["_part_id_num"].between(MODELING_START_PART_ID, MODELING_END_PART_ID)
].copy()

print("Rows after modeling-period filter:", len(df_model))
print(
    "Modeling part range:",
    df_model["_part_id_num"].min(),
    "→",
    df_model["_part_id_num"].max()
)


# ============================================================
# 2. SORT FIRST BY IDS
# ============================================================

sort_cols = [
    "student_id",
    "degree_id",
    "_part_id_num",
    "course_id",
]

df_model = (
    df_model
    .sort_values(sort_cols, kind="mergesort", na_position="last")
    .reset_index(drop=True)
)

print("Data sorted by student_id + degree_id + part_id + course_id")


# ============================================================
# 3. CREATE TARGETS
# ============================================================

# M1 target: pass/fail
# You decided that pass/fail can be known from final_mark.
# IMPORTANT: final_mark will NOT enter X.
PASS_MARK = 50

df_model["final_mark"] = pd.to_numeric(df_model["final_mark"], errors="coerce")

# Drop rows with missing final_mark target
df_model = df_model[df_model["final_mark"].notna()].copy()

df_model["y_pass"] = df_model["final_mark"].ge(PASS_MARK).astype("int8")

# M2 target: final mark regression for now
# Later, if you use true GPA points, replace y_mark with that target.
df_model["y_mark"] = df_model["final_mark"].copy()

print("Pass/fail distribution:")
print(df_model["y_pass"].value_counts(dropna=False))

print("Final mark target summary:")
display(df_model["y_mark"].describe())


# ============================================================
# 4. TEMPORAL SPLIT BY part_id
# ============================================================

TRAIN_START_PART_ID = 20051
TRAIN_END_PART_ID = 20223

VALID_START_PART_ID = 20231
VALID_END_PART_ID = 20233

TEST_START_PART_ID = 20241
TEST_END_PART_ID = 20251

train_df = df_model[
    df_model["_part_id_num"].between(TRAIN_START_PART_ID, TRAIN_END_PART_ID)
].copy()

valid_df = df_model[
    df_model["_part_id_num"].between(VALID_START_PART_ID, VALID_END_PART_ID)
].copy()

test_df = df_model[
    df_model["_part_id_num"].between(TEST_START_PART_ID, TEST_END_PART_ID)
].copy()

print("Train shape:", train_df.shape)
print("Valid shape:", valid_df.shape)
print("Test shape:", test_df.shape)

print("Train part range:", train_df["part_id"].min(), "→", train_df["part_id"].max())
print("Valid part range:", valid_df["part_id"].min(), "→", valid_df["part_id"].max())
print("Test part range:", test_df["part_id"].min(), "→", test_df["part_id"].max())

# Optional sanity check: make sure no overlap exists
assert train_df["_part_id_num"].max() < valid_df["_part_id_num"].min(), "Train/valid temporal overlap detected"
assert valid_df["_part_id_num"].max() < test_df["_part_id_num"].min(), "Valid/test temporal overlap detected"


# ============================================================
# 5. DEFINE UNWANTED COLUMNS
# ============================================================

unwanted_columns = [
    # IDs / tracking columns - keep for audit, not for ML features
    "student_course_id",
    "student_id",
    "course_id",
    "degree_id",
    "faculty_id",
    "grade_id",
    "student_status_id",
    "part_id",
    "_part_id_num",
    "start_part_id",
    "degree_course_key",

    # Targets / leakage columns
    "final_mark",
    "gpa_points",
    "semester_pass_credits",
    "is_interruption_semester",
    "y_pass",
    "y_mark",

    # Outlier filter / audit flags, not ML features
    "over_policy_semester_credits",
    "over_policy_semester_courses",
    "exclude_over_policy_semester",

    # Raw / redundant / helper columns
    "prev_gpa_points",
    "prev_gpa_points_zero",
    "prev_gpa_no_previous_active_semester",
    "last_valid_gpa_before_current_semester",
    "first_semester_gpa_baseline",
    "used_first_semester_gpa_baseline",
    "no_previous_progress",

    # Raw cumulative values replaced by better versions
    "total_pass_credits",
    "total_fail_credits",

    # Raw text / redundant time columns
    "start_level_name_pl",
    "part_year",
    "start_year",
    "start_semester",
]

unwanted_columns_existing = [col for col in unwanted_columns if col in train_df.columns]

print("Unwanted columns to eject/drop:")
print(unwanted_columns_existing)


# ============================================================
# 6. EJECT UNWANTED COLUMNS BEFORE DROPPING
# ============================================================

# These are kept for audit/debugging.
X_train_ejected_columns = train_df[
    [col for col in unwanted_columns_existing if col in train_df.columns]
].copy()

X_valid_ejected_columns = valid_df[
    [col for col in unwanted_columns_existing if col in valid_df.columns]
].copy()

X_test_ejected_columns = test_df[
    [col for col in unwanted_columns_existing if col in test_df.columns]
].copy()

print("Ejected train columns shape:", X_train_ejected_columns.shape)
print("Ejected valid columns shape:", X_valid_ejected_columns.shape)
print("Ejected test columns shape:", X_test_ejected_columns.shape)


# ============================================================
# 7. BUILD y_train / y_valid / y_test
# ============================================================

# M1 classification target
y_train_pass = train_df["y_pass"].copy()
y_valid_pass = valid_df["y_pass"].copy()
y_test_pass = test_df["y_pass"].copy()

# M2 regression target
y_train_mark = train_df["y_mark"].copy()
y_valid_mark = valid_df["y_mark"].copy()
y_test_mark = test_df["y_mark"].copy()


# ============================================================
# 8. BUILD X_train / X_valid / X_test BY DROPPING UNWANTED COLUMNS
# ============================================================

X_train = train_df.drop(columns=unwanted_columns_existing, errors="ignore").copy()
X_valid = valid_df.drop(columns=unwanted_columns_existing, errors="ignore").copy()
X_test = test_df.drop(columns=unwanted_columns_existing, errors="ignore").copy()

# Force validation/test columns to match train columns exactly
X_valid = X_valid.reindex(columns=X_train.columns)
X_test = X_test.reindex(columns=X_train.columns)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("X_test shape:", X_test.shape)


# ============================================================
# 9. FINAL SAFETY CHECKS
# ============================================================

blocked_columns = [
    "student_id",
    "course_id",
    "degree_id",
    "part_id",
    "_part_id_num",
    "degree_course_key",

    "final_mark",
    "gpa_points",
    "semester_pass_credits",
    "is_interruption_semester",
    "y_pass",
    "y_mark",

    "prev_gpa_points",
    "total_pass_credits",
    "total_fail_credits",

    "part_year",
    "start_year",
    "start_semester",
]

blocked_found_train = [col for col in blocked_columns if col in X_train.columns]
blocked_found_valid = [col for col in blocked_columns if col in X_valid.columns]
blocked_found_test = [col for col in blocked_columns if col in X_test.columns]

assert not blocked_found_train, f"Blocked columns still found in X_train: {blocked_found_train}"
assert not blocked_found_valid, f"Blocked columns still found in X_valid: {blocked_found_valid}"
assert not blocked_found_test, f"Blocked columns still found in X_test: {blocked_found_test}"

assert len(X_train) == len(y_train_pass), "X_train and y_train_pass length mismatch"
assert len(X_valid) == len(y_valid_pass), "X_valid and y_valid_pass length mismatch"
assert len(X_test) == len(y_test_pass), "X_test and y_test_pass length mismatch"

assert len(X_train) == len(y_train_mark), "X_train and y_train_mark length mismatch"
assert len(X_valid) == len(y_valid_mark), "X_valid and y_valid_mark length mismatch"
assert len(X_test) == len(y_test_mark), "X_test and y_test_mark length mismatch"

assert list(X_train.columns) == list(X_valid.columns), "Train/valid columns do not match"
assert list(X_train.columns) == list(X_test.columns), "Train/test columns do not match"

print("All checks passed.")

print("Final X_train columns:")
print(X_train.columns)

print("Final X_valid columns:")
print(X_valid.columns)

print("Final X_test columns:")
print(X_test.columns)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. START FROM CLEAN PRIMARY DATA
# ============================================================

df_dist = df.copy()

# Remove over-policy semesters if the flag exists
if "exclude_over_policy_semester" in df_dist.columns:
    df_dist = df_dist[df_dist["exclude_over_policy_semester"].eq(False)].copy()

# Make numeric part_id for correct chronological sorting
df_dist["_part_id_num"] = pd.to_numeric(df_dist["part_id"], errors="coerce")

# Drop rows with invalid part_id
df_dist = df_dist[df_dist["_part_id_num"].notna()].copy()

df_dist["_part_id_num"] = df_dist["_part_id_num"].astype(int)

# Optional: extract year and semester if not already available
df_dist["part_year_check"] = df_dist["_part_id_num"] // 10
df_dist["part_semester_check"] = df_dist["_part_id_num"] % 10

print("Data shape for distribution:", df_dist.shape)
print("Part range:", df_dist["_part_id_num"].min(), "→", df_dist["_part_id_num"].max())

In [ ]:
PASS_MARK = 50

df_dist["_pass_tmp"] = df_dist["final_mark"].ge(PASS_MARK).astype(int)
df_dist["_fail_tmp"] = 1 - df_dist["_pass_tmp"]

In [ ]:
part_distribution = (
    df_dist
    .groupby("_part_id_num")
    .agg(
        row_count=("student_course_id", "count"),
        unique_students=("student_id", "nunique"),
        unique_courses=("course_id", "nunique"),
        unique_degrees=("degree_id", "nunique"),
        avg_final_mark=("final_mark", "mean"),
        median_final_mark=("final_mark", "median"),
        pass_rate=("_pass_tmp", "mean"),
        fail_rate=("_fail_tmp", "mean"),
        fail_count=("_fail_tmp", "sum"),
        min_mark=("final_mark", "min"),
        max_mark=("final_mark", "max"),
    )
    .reset_index()
    .rename(columns={"_part_id_num": "part_id"})
)

# Sort chronologically
part_distribution = part_distribution.sort_values("part_id").reset_index(drop=True)

# Add cumulative info
total_rows = part_distribution["row_count"].sum()

part_distribution["cumulative_rows"] = part_distribution["row_count"].cumsum()
part_distribution["cumulative_row_pct"] = (
    part_distribution["cumulative_rows"] / total_rows
)

part_distribution["remaining_rows_after_part"] = (
    total_rows - part_distribution["cumulative_rows"]
)

part_distribution["remaining_row_pct_after_part"] = (
    part_distribution["remaining_rows_after_part"] / total_rows
)

display(part_distribution)

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(part_distribution["part_id"].astype(str), part_distribution["row_count"], marker="o")
plt.xticks(rotation=90)
plt.title("Rows per Semester")
plt.xlabel("part_id")
plt.ylabel("Row count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 5))
plt.plot(part_distribution["part_id"].astype(str), part_distribution["unique_students"], marker="o")
plt.xticks(rotation=90)
plt.title("Unique Students per Semester")
plt.xlabel("part_id")
plt.ylabel("Unique students")
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 5))
plt.plot(part_distribution["part_id"].astype(str), part_distribution["fail_rate"], marker="o")
plt.xticks(rotation=90)
plt.title("Fail Rate per Semester")
plt.xlabel("part_id")
plt.ylabel("Fail rate")
plt.tight_layout()
plt.show()

In [ ]:
TRAIN_START_PART_ID

In [ ]:
X_train_ejected_columns

In [ ]:
X_train

In [ ]:
X_train[(X_train['prev_gpa_points_clean']==0)&(X_train['is_first_active_semester']==0)]

In [ ]:
X_train[(X_train['prev_gpa_points_clean']==0)&(X_train['start_level_ord']==2)]

In [ ]:
df1=pd.read_parquet(r'D:\AI\Real projects\Academic_Advisor\data\preprocessed\V_ADD_STUDENT_DEGREE_STATUS\clean_v_add_student_degree_status.parquet')

In [ ]:
df1[df1['student_id']==10428.111]

In [ ]:
df2=pd.read_parquet(r'D:\AI\Real projects\Academic_Advisor\data\preprocessed\V_CRG_STUDENT_COURSE\clean_v_crg_student_course.parquet')

In [ ]:
df2[df2['student_id']==10428.111]